# Transaction Foundation Model on Ray — Part 9: Scaling up, and the bottlenecks you'll hit

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~15 min

---

The rest of the series said "same code, change a config." That's true — but it undersells the point. Scaling up isn't free: you hit real walls, and what makes Ray and Anyscale worth it is that each wall has a fix that *doesn't* mean rewriting your pipeline. This notebook walks four bottlenecks you actually hit going from `mini` toward the full dataset and beyond, and shows the fix for each.

Some of these walls only appear with real GPUs or multiple nodes, which a CPU walkthrough can't conjure. Where that's the case, the heavy run is a script you launch as an Anyscale Job and watch in the dashboard — and **every number in this notebook is measured on a real run, never invented.**

In [ ]:
import sys, os, time, shutil

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir

paths = artifact_paths(get_demo_base_dir(), "mini")   # Sections 3-4 reuse the mini artifacts from Parts 3-5

In [ ]:
# Sections 3-4 reuse the mini tokenized windows from Parts 3-5. Part 8's fused
# pipeline streams those through memory (it never writes tokenized Parquet), so
# if you ran Part 8 they may be absent — regenerate them here if needed.
import subprocess

if not os.path.exists(paths["tokenized_eval"]):
    print("tokenized windows not found — regenerating from cached raw data (~2 min)...")
    subprocess.run([sys.executable, "scripts/02_tokenize.py", "--scale", "mini"], check=True)
else:
    print("tokenized windows present — reusing.")

## Bottleneck 1 — the data outgrows one node

The first wall isn't training — it's the data. The synthetic generator from Part 1 (`generate_transactions`) builds the whole dataset in one process and returns one DataFrame; that's fine for thousands of cards and impossible for the hundreds of millions of rows you'd want to actually exercise a foundation model. You hit it *before you can even start*.

The fix is to make data generation itself a distributed job. `generate_dataset_distributed` maps the same per-card generator over blocks of card ids with **Ray Data**: each block is produced on whatever worker has a slot, the result streams to sharded Parquet, and no single process ever holds the dataset. The same shape — `ray.data.range(...).map_batches(...).write_parquet(...)` — is how you'd generate, read, or transform data of any size.

In [ ]:
from src.generate_data import generate_dataset_distributed

scale_dir = os.path.join(get_demo_base_dir(), "scale_demo")
raw_big = os.path.join(scale_dir, "raw.parquet")
shutil.rmtree(scale_dir, ignore_errors=True)

t0 = time.perf_counter()
generate_dataset_distributed(
    raw_big, os.path.join(scale_dir, "splits.json"),
    num_cards=10_000, num_blocks=64,        # ~300k txns; bump num_cards on a bigger cluster
)
n = ray.data.read_parquet(raw_big).count()
print(f"\ngenerated {n:,} transactions in {time.perf_counter() - t0:.0f}s — "
      f"sharded across the cluster, never held in one process")

That ran across the cluster on CPU. To feel the wall and the scale-out, generate a *much* bigger set as a job — the cluster autoscales CPU nodes to produce it and scales back to zero:

```bash
python scripts/generate_big.py --num-cards 1000000          # ~30M+ txns, on the workspace
anyscale job submit --config-file job_config.yaml -- python scripts/generate_big.py --num-cards 20000000
```

The single-node generator (`generate_transactions`) on the same card count would hold every row in one Python process and OOM the head node — which is exactly why generation had to go distributed.

## Bottleneck 2 — the GPU sits idle, starved by the CPU

The embedding stage (Part 5) is heterogeneous: **CPU** reads and tokenizes the data, **GPU** runs the forward pass. The trap is scaling them together. If you add GPU workers but the CPU side can't tokenize fast enough to keep them fed, your expensive accelerators sit idle — you're paying GPU prices for the dashboard to show 20% utilization.

It's a throughput-matching problem: if one CPU core tokenizes *r_cpu* rows/s and one GPU embeds *r_gpu* rows/s, you need roughly `r_gpu / r_cpu` CPU cores feeding each GPU. A fixed, homogeneous cluster can't satisfy that ratio — and this is where Anyscale earns its keep over a single fixed pool: `job_config.yaml` declares **separate CPU and GPU worker groups**, and the autoscaler sizes each independently so the CPU group scales out until the GPU group is saturated.

```yaml
worker_nodes:
  - name: gpu-1x      # embed/pretrain forward passes
    instance_type: g4dn.xlarge
    min_nodes: 0
    max_nodes: 4
  - name: cpu-workers # the tokenize shuffle that *feeds* the GPUs
    instance_type: m5.4xlarge
    min_nodes: 0
    max_nodes: 1
```

To see it: run the embedding stage on a GPU and watch GPU utilization in the Ray dashboard. Starve it (one CPU worker) and utilization sags between batches; give the CPU group room and it climbs toward 100%.

```bash
anyscale job submit --config-file job_config.yaml -- python scripts/04_extract_embeddings.py --scale small --use-gpu
```

*(This is the one wall a CPU walkthrough can't show directly — there's no GPU here to leave idle — so its numbers live in the dashboard of the job above, not in this notebook.)*

## Bottleneck 3 — every stage boundary hits disk

The step-by-step notebooks wrote Parquet between every stage so each was independently runnable. At scale that's a tax: every boundary pays a full write **and** read, and the training stage re-reads its whole input *from storage on every epoch*. With a 20-epoch run over GB of windows, that re-read dominates the wall clock.

Ray Data's fix is to keep the intermediate in the cluster: **materialize it once in the object store** and iterate from memory. The cell below measures it directly — iterating the training windows three times, re-reading Parquet each pass versus reading once into the object store:

In [ ]:
def iterate(ds, epochs=3):
    t = time.perf_counter()
    for _ in range(epochs):
        for _batch in ds.iter_batches(batch_size=512):
            pass
    return time.perf_counter() - t

p = paths["tokenized_pretrain"]
t_parquet = iterate(ray.data.read_parquet(p))                 # lazy: re-reads Parquet every epoch
t_objstore = iterate(ray.data.read_parquet(p).materialize())  # materialized once, iterated from memory
print(f"3 epochs from Parquet:       {t_parquet:5.1f}s")
print(f"3 epochs from object store:  {t_objstore:5.1f}s")
print(f"-> {t_parquet / t_objstore:.1f}x faster, and the gap grows with epochs and data size")

Even at this tiny scale, iterating from the object store is **several times faster** (2–6× across runs, depending on cache state). That's not a micro-optimization you hand-code — it's what `scripts/run_pipeline.py` already does: it materializes the pretrain windows in the object store and hands them to Ray Train as a live dataset, and streams the eval windows straight from the CPU tokenizer into the GPU embedding actors, so the intermediates never touch disk. The step-by-step scripts (`02`–`05`) take the Parquet-at-every-boundary path; `run_pipeline.py` is the fused one. Same stages, same results — one keeps the data moving through cluster memory, and the gap widens with epochs and data size.

## Bottleneck 4 — latency collapses under load

Online fraud scoring has a hard latency budget — an authorization decision has to come back in tens to low-hundreds of milliseconds. A single model replica handles one request at a time; the moment concurrent traffic arrives, requests queue and tail latency blows past the SLA. This is a serving bottleneck, distinct from batch throughput.

Ray Serve's fix is to **autoscale replicas** on a target concurrency (and batch requests for accelerator efficiency). The cell below load-tests the deployment under 24 concurrent clients at a fixed 1 replica, then at 4, and reports p50/p99 latency and throughput. (`src/serve.py` ships an `autoscaling_config` of 1→4 replicas that does this automatically under real traffic; here we pin the count to make the contrast clean.)

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor
from ray import serve
from src.serve import TransactionEmbeddingService
from src.utils import sample_serve_payload

payload = sample_serve_payload(paths["tokenized_eval"])
SLA_MS = 200

def load_test(n_replicas, n_requests=400, concurrency=24):
    app = TransactionEmbeddingService.options(
        num_replicas=n_replicas, autoscaling_config=None,
    ).bind(checkpoint_dir=paths["checkpoint"])
    serve.run(app, name="txn-fm")
    for _ in range(10):                                  # warm up
        requests.post("http://localhost:8000/", json=payload, timeout=30)

    def one(_):
        t = time.perf_counter()
        requests.post("http://localhost:8000/", json=payload, timeout=60)
        return (time.perf_counter() - t) * 1000

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        lat = sorted(ex.map(one, range(n_requests)))
    wall = time.perf_counter() - t0
    p50, p99 = lat[len(lat) // 2], lat[int(len(lat) * 0.99) - 1]
    verdict = "under" if p99 < SLA_MS else "OVER"
    print(f"{n_replicas} replica(s): p50={p50:5.0f}ms  p99={p99:5.0f}ms  "
          f"({verdict} {SLA_MS}ms SLA)  throughput={n_requests / wall:3.0f} req/s")
    serve.shutdown()

load_test(1)
load_test(4)

Expect the single replica to **breach** the 200 ms p99 SLA under load (300+ ms in our runs) while 4 replicas come in **under** it (~170 ms) at roughly 2× the throughput. In production you don't pin the count — `autoscaling_config` adds replicas as `target_ongoing_requests` is exceeded and removes them when traffic drops, so you hold the SLA without paying for idle replicas off-peak. For accelerator-backed serving, `@serve.batch` coalesces concurrent requests into one forward pass on top of this.

## Takeaways

Scaling an ML pipeline is a sequence of bottlenecks, and the reason to build on Ray and Anyscale is that each one has a fix that's a config or a primitive — not a rewrite:

| Wall you hit | Where | The Ray/Anyscale fix |
|---|---|---|
| Data won't fit one node | data prep, loading | Ray Data distributed streaming + node autoscaling |
| GPUs idle, starved by CPU | batch embedding | independent CPU/GPU worker-group autoscaling |
| Stage boundaries hit disk | multi-stage pipeline | object-store materialization + streaming (`run_pipeline.py`) |
| Tail latency under load | online serving | Ray Serve replica autoscaling + request batching |

"Change one config and it scales" is the headline. The substance underneath it is that the framework already has an answer for each wall you'll actually hit — which is what makes the headline true instead of a sales pitch.

That's the series: a transaction foundation model built, evaluated, served, and scaled — the same code from a laptop to a multi-node GPU cluster, on Ray and Anyscale.